## CELL 1 — Install & Import Library
*PERSIAPAN*

In [ ]:
!pip -q install openpyxl scipy seaborn

import os, re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import (silhouette_score, silhouette_samples,
                             davies_bouldin_score, calinski_harabasz_score,
                             adjusted_rand_score)
import joblib

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

## CELL 2 — Konfigurasi Path, Seed, dan Folder Output
*PERSIAPAN*

In [ ]:
FILE_PATH = (
    "/content/Dataset___Pola_Pengeluaran_Mahasiswa_Tanjung_Pinang.xlsx"
)
OUT_DIR = "/content/hasil"
PNG_DIR = os.path.join(OUT_DIR, "hasil_png")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PNG_DIR, exist_ok=True)

# Bersihkan PNG dari eksekusi sebelumnya agar ZIP hanya berisi hasil run terbaru.
for nama_file in os.listdir(PNG_DIR):
    if nama_file.lower().endswith(".png"):
        os.remove(os.path.join(PNG_DIR, nama_file))

# Simpan otomatis setiap grafik yang ditampilkan.
PNG_COUNTER = 0

def _nama_png(prefix="output"):
    global PNG_COUNTER
    PNG_COUNTER += 1
    return os.path.join(PNG_DIR, f"{PNG_COUNTER:03d}_{prefix}.png")

_original_plt_show = plt.show

def _show_dan_simpan_grafik(*args, **kwargs):
    nomor_fig = list(plt.get_fignums())
    for nomor in nomor_fig:
        fig = plt.figure(nomor)
        fig.savefig(_nama_png("grafik"), dpi=200, bbox_inches="tight")
    return _original_plt_show(*args, **kwargs)

plt.show = _show_dan_simpan_grafik

# Simpan setiap DataFrame/Series yang ditampilkan sebagai gambar PNG.
from IPython.display import display as _display_asli

def _simpan_tabel_png(obj):
    if isinstance(obj, pd.Series):
        tabel = obj.to_frame()
    elif isinstance(obj, pd.DataFrame):
        tabel = obj.copy()
    else:
        return

    if tabel.empty:
        return

    # Format tampilan agar tabel tetap terbaca pada gambar.
    tabel = tabel.copy()
    for kolom in tabel.columns:
        if pd.api.types.is_numeric_dtype(tabel[kolom]):
            tabel[kolom] = tabel[kolom].map(
                lambda nilai: f"{nilai:,.4f}" if isinstance(nilai, (float, np.floating))
                else f"{nilai:,}" if isinstance(nilai, (int, np.integer))
                else nilai
            )

    n_baris, n_kolom = tabel.shape
    lebar = max(8, min(24, 2.0 + n_kolom * 1.35))
    tinggi = max(2.5, min(30, 1.5 + n_baris * 0.32))

    fig, ax = plt.subplots(figsize=(lebar, tinggi))
    ax.axis("off")
    tabel_gambar = ax.table(
        cellText=tabel.astype(str).values,
        colLabels=[str(k) for k in tabel.columns],
        rowLabels=[str(i) for i in tabel.index] if not isinstance(tabel.index, pd.RangeIndex) else None,
        cellLoc="center",
        loc="center"
    )
    tabel_gambar.auto_set_font_size(False)
    tabel_gambar.set_fontsize(7 if n_kolom > 8 else 8)
    tabel_gambar.scale(1, 1.25)
    fig.tight_layout()
    fig.savefig(_nama_png("tabel"), dpi=200, bbox_inches="tight")
    plt.close(fig)

def display(*objs, **kwargs):
    for obj in objs:
        _simpan_tabel_png(obj)
    return _display_asli(*objs, **kwargs)

SEED = 42
HARI = ["senin", "selasa", "rabu", "kamis", "jumat", "sabtu", "minggu"]
print("File ada :", os.path.exists(FILE_PATH))
print("Output   :", OUT_DIR)
print("Folder PNG:", PNG_DIR)


## CELL 3 — I.4 Lini Waktu Pelaksanaan (Gantt Chart & PIC)
*BAB I - BUSINESS UNDERSTANDING*\n\n*Tanggal mulai dan selesai diselaraskan dengan rentang pertemuan pada file Jadwal Penambangan Data. Notebook berfokus pada proses analisis dan pemodelan, tanpa tahap pembuatan aplikasi web.*

In [ ]:
kegiatan = [
    # (kegiatan, PIC, mulai, selesai) — tanggal mengikuti jadwal perkuliahan Data Mining.
    ("Business Understanding & penyusunan proposal", "Rydhoi Trimaniel Lase", "2026-09-08", "2026-09-19"),
    ("Data Understanding", "Riki Andika Saputra", "2026-09-15", "2026-09-18"),
    ("EDA: data deskriptif & eksplorasi", "Riki Andika Saputra", "2026-09-22", "2026-09-25"),
    ("EDA: visualisasi data", "Riki Andika Saputra", "2026-09-29", "2026-10-02"),
    ("Data Selection, Transformation & Feature Engineering", "Ardiansyah Rizki Khairi Ali", "2026-10-13", "2026-10-16"),
    ("Pemodelan K-Means", "Aldi Saputra", "2026-10-27", "2026-10-30"),
    ("Evaluasi & pengukuran model", "Ikhbal Maulana", "2026-11-10", "2026-11-13"),
    ("Penyusunan laporan & video presentasi final", "Rydhoi Trimaniel Lase", "2026-11-17", "2026-11-27"),
]
gantt = pd.DataFrame(kegiatan, columns=["kegiatan", "pic", "mulai", "selesai"])
gantt["mulai"] = pd.to_datetime(gantt["mulai"])
gantt["selesai"] = pd.to_datetime(gantt["selesai"])
gantt["durasi"] = (gantt["selesai"] - gantt["mulai"]).dt.days + 1

fig, ax = plt.subplots(figsize=(12, 4.8))
for i, r in gantt.iterrows():
    ax.barh(i, r["durasi"], left=mdates.date2num(r["mulai"]), color="#2E75B6")
    ax.text(mdates.date2num(r["selesai"]) + 1.2, i, r["pic"], va="center", fontsize=8)
ax.set_yticks(range(len(gantt)))
ax.set_yticklabels(gantt["kegiatan"])
ax.invert_yaxis()
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.set_xlim(mdates.date2num(pd.Timestamp("2026-09-06")), mdates.date2num(pd.Timestamp("2026-11-30")))
ax.set_title("Jadwal Pelaksanaan Project Penambangan Data (Mengacu Jadwal Perkuliahan)")
ax.set_xlabel("Periode pelaksanaan")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/gantt_chart.png", bbox_inches="tight")
plt.show()
display(gantt)


## CELL 4 — II.1 Memuat Dataset Hasil Google Form
*BAB II - DATA UNDERSTANDING*

In [ ]:
df_raw = pd.read_excel(FILE_PATH, sheet_name=0)
print("Jumlah baris (responden):", df_raw.shape[0])
print("Jumlah kolom            :", df_raw.shape[1])
display(df_raw.head())

## CELL 5 — II.2 Struktur Kolom & Tipe Data
*BAB II - DATA UNDERSTANDING*

In [ ]:
info = pd.DataFrame({
    "kolom": df_raw.columns,
    "dtype": df_raw.dtypes.astype(str).values,
    "non_null": df_raw.notna().sum().values,
    "null": df_raw.isna().sum().values,
    "n_unik": df_raw.nunique().values,
    "contoh": [str(df_raw[c].dropna().iloc[0])[:35] if df_raw[c].notna().any() else "-"
               for c in df_raw.columns],
})
display(info)

## CELL 6 — II.2 Jenis Data (Skala Pengukuran per Variabel)
*BAB II - DATA UNDERSTANDING*

In [ ]:
jenis_data = pd.DataFrame([
    ["Cap waktu", "Waktu pengisian",                 "Interval (datetime)", "Tidak dipakai model"],
    ["Nama(bisa samaran)", "Identitas responden",             "Nominal (teks)",      "Dibuang, diganti ID"],
    ["Nama kampus", "Asal kampus (opsional)",          "Nominal (kategorik)", "Tidak dipakai model"],
    ["Semester saat ini", "Semester responden",              "Ordinal (numerik)",   "Profil cluster"],
    ["IPK terakhir", "Prestasi akademik",               "Rasio (numerik)",     "Fitur model"],
    ["Total pengeluaran <hari> (Rp)", "Pengeluaran harian (7 kolom)", "Rasio (numerik)", "Fitur model (dijumlah)"],
    ["Rincian pengeluaran <hari>", "Rincian belanja bebas (7 kolom)", "Teks bebas", "Dikategorikan (FE)"],
    ["Durasi kuliah", "Jam/minggu",                      "Rasio (numerik)",     "Fitur model"],
    ["Durasi belajar mandiri", "Jam/minggu",                      "Rasio (numerik)",     "Fitur model"],
    ["Durasi organisasi", "Jam/minggu",                      "Rasio (numerik)",     "Fitur model"],
    ["Durasi bekerja", "Jam/minggu",                      "Rasio (numerik)",     "Fitur model"],
    ["Durasi kegiatan lain", "Jam/minggu",                      "Rasio (numerik)",     "Fitur model"],
    ["Kolom 20", "Kolom kosong bawaan form",        "-",                   "Dibuang"],
], columns=["variabel", "keterangan", "jenis/skala data", "peran"])
display(jenis_data)

## CELL 7 — II.3 EDA - Ringkasan Statistik Deskriptif
*BAB II - DATA UNDERSTANDING*

In [ ]:
kol_total = [c for c in df_raw.columns if c.startswith("Total pengeluaran")]
kol_jam   = [c for c in df_raw.columns if c.startswith("Total durasi")]
kol_num   = ["Semester saat ini", "IPK terakhir"] + kol_total + kol_jam

display(df_raw[kol_num].describe().T.round(2))
display(df_raw[["Nama kampus"]].describe())
print("Skewness:")
display(df_raw[kol_num].skew().round(2).to_frame("skew").T)

## CELL 8 — II.3 EDA - Missing Value & Kualitas Data
*BAB II - DATA UNDERSTANDING*

In [ ]:
mv = df_raw.isna().sum().to_frame("jumlah_kosong")
mv["persen"] = (mv["jumlah_kosong"] / len(df_raw) * 100).round(1)
display(mv[mv["jumlah_kosong"] > 0])

nama_ganda = df_raw["Nama(bisa samaran)"].str.strip().str.lower().duplicated(keep=False).sum()
print("Baris dgn nama sama (BUKAN duplikat) :", nama_ganda)
print("Baris identik 100% (duplikat sungguhan):", df_raw.duplicated().sum())
print("Rentang IPK :", df_raw["IPK terakhir"].min(), "-", df_raw["IPK terakhir"].max())
print("Jam total/minggu maks:", df_raw[kol_jam].sum(axis=1).max(), "(batas wajar 168)")

## CELL 9 — II.3 EDA - Visualisasi Distribusi IPK & Semester
*BAB II - DATA UNDERSTANDING*

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df_raw["IPK terakhir"], bins=15, kde=True, ax=ax[0], color="#2E75B6")
ax[0].set_title("Distribusi IPK")
sns.countplot(x="Semester saat ini", data=df_raw, ax=ax[1], color="#70AD47")
ax[1].set_title("Jumlah Responden per Semester")
plt.tight_layout()
plt.show()

## CELL 10 — II.3 EDA - Visualisasi Pengeluaran Harian
*BAB II - DATA UNDERSTANDING*

In [ ]:
ren = {f"Total pengeluaran {h.capitalize()} (Rp)": h.capitalize() for h in HARI}
harian = df_raw[kol_total].rename(columns=ren)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
sns.boxplot(data=harian, ax=ax[0], palette="Blues")
ax[0].set_title("Boxplot Pengeluaran per Hari (Rp)")
ax[0].set_ylabel("Rp")
harian.mean().plot(kind="bar", ax=ax[1], color="#ED7D31")
ax[1].set_title("Rata-rata Pengeluaran per Hari (Rp)")
ax[1].tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

print("Jumlah hari dengan pengeluaran Rp0:", (harian == 0).sum().sum(),
      "dari", harian.size, "hari-responden")

## CELL 11 — II.3 EDA - Visualisasi Durasi Aktivitas Mingguan
*BAB II - DATA UNDERSTANDING*

In [ ]:
jam = df_raw[kol_jam].copy()
jam.columns = ["Kuliah", "Belajar mandiri", "Organisasi", "Bekerja", "Kegiatan lain"]

jam.hist(bins=12, figsize=(14, 6), color="#5B9BD5")
plt.suptitle("Histogram Durasi Aktivitas per Minggu (jam)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.boxplot(data=jam, palette="Set2")
plt.title("Boxplot Durasi Aktivitas per Minggu (jam)")
plt.show()

## CELL 12 — II.3 EDA - Pola Awal Pengeluaran vs IPK (Referensi Sketsa)
*BAB II - DATA UNDERSTANDING*

In [ ]:
tmp = df_raw.copy()
tmp["pengeluaran_mingguan"] = tmp[kol_total].sum(axis=1)

plt.figure(figsize=(8, 5.5))
sns.scatterplot(x="pengeluaran_mingguan", y="IPK terakhir", data=tmp,
                color="#2E75B6", s=45, alpha=.8)
plt.xlabel("Total pengeluaran per minggu (Rp)")
plt.ylabel("IPK")
plt.title("Sebaran Awal: Pengeluaran (sumbu X) vs IPK (sumbu Y)")
plt.show()

r, p = stats.pearsonr(tmp["pengeluaran_mingguan"], tmp["IPK terakhir"])
print(f"Korelasi Pearson pengeluaran-IPK: r={r:.3f}, p={p:.3f}")

## CELL 13 — II.3 EDA - Pola Teks Rincian Pengeluaran (Frekuensi Kata)
*BAB II - DATA UNDERSTANDING*

In [ ]:
kol_rinci = [c for c in df_raw.columns if c.startswith("Rincian")]
teks = pd.concat([df_raw[c] for c in kol_rinci]).astype(str).str.lower()
kata = Counter(w for t in teks for w in re.findall(r"[a-z]{3,}", t))
stop = {"tidak", "ada", "pengeluaran", "dan", "yang", "untuk"}
top = pd.DataFrame([(k, v) for k, v in kata.most_common(60) if k not in stop][:25],
                   columns=["kata", "frekuensi"])

plt.figure(figsize=(9, 5))
sns.barplot(y="kata", x="frekuensi", data=top, color="#5B9BD5")
plt.title("25 Kata Terbanyak pada Rincian Pengeluaran")
plt.tight_layout()
plt.show()

print("Contoh format penulisan rincian (bebas):")
for t in teks.sample(6, random_state=SEED):
    print(" -", repr(t))

## CELL 14 — II.3 EDA - Deteksi Permasalahan Data (Ringkasan)
*BAB II - DATA UNDERSTANDING*

In [ ]:
def n_outlier_iqr(s):
    q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    return int(((s < q1 - 1.5 * i) | (s > q3 + 1.5 * i)).sum())

masalah = pd.DataFrame({
    "variabel": ["pengeluaran_mingguan"] + list(jam.columns) + ["IPK"],
    "n_outlier_IQR": [n_outlier_iqr(tmp["pengeluaran_mingguan"])]
                     + [n_outlier_iqr(jam[c]) for c in jam.columns]
                     + [n_outlier_iqr(df_raw["IPK terakhir"])],
})
display(masalah)

print("\nDaftar permasalahan data yang ditemukan:")
print("1. 'Kolom 20' kosong seluruhnya            -> dibuang")
print("2. Nama bebas/samaran, ada nama yang sama  -> diganti ID unik, BUKAN duplikat")
print("3. Rincian pengeluaran berupa teks bebas   -> parsing & kategorisasi")
print("4. Format nominal beragam (10k, 40.000)    -> normalisasi angka")
print("5. Total harian vs jumlah rincian          -> validasi konsistensi")
print("6. Outlier pengeluaran (nilai ekstrem)     -> penanganan outlier")
print("7. Skala fitur berbeda (Rp vs jam vs IPK)  -> standardisasi")

## CELL 15 — III.1 Seleksi Kolom & Penyederhanaan Nama Kolom
*BAB III - PREPROCESS*

In [ ]:
df = df_raw.copy()

ren = {
    "Cap waktu": "timestamp",
    "Nama(bisa samaran)": "nama",
    "Nama kampus": "kampus",
    "Semester saat ini": "semester",
    "IPK terakhir": "ipk",
    "Total durasi kuliah/perkuliahan selama seminggu (jam)": "jam_kuliah",
    "Total durasi belajar mandiri selama seminggu (jam)": "jam_belajar",
    "Total durasi kegiatan organisasi selama seminggu (jam)": "jam_organisasi",
    "Total durasi bekerja selama seminggu (jam)": "jam_kerja",
    "Total durasi kegiatan lain selama seminggu (jam)": "jam_lain",
}
for h in HARI:
    ren[f"Total pengeluaran {h.capitalize()} (Rp)"] = f"total_{h}"
    ren[f"Rincian pengeluaran {h.capitalize()}"] = f"rincian_{h}"
df = df.rename(columns=ren)

kolom_kosong = [c for c in df.columns if df[c].isna().all()]
print("Kolom kosong dibuang:", kolom_kosong)
df = df.drop(columns=kolom_kosong)
print("Bentuk data:", df.shape)

## CELL 16 — III.2 Anonimisasi: Hapus Nama, Buat ID Unik
*BAB III - PREPROCESS*

In [ ]:
df = df.sort_values("timestamp").reset_index(drop=True)
df.insert(0, "id_mhs", [f"MHS{i:03d}" for i in range(1, len(df) + 1)])

print("Nama sama (tetap dianggap responden berbeda):")
display(df[df["nama"].str.lower().str.strip().duplicated(keep=False)]
        [["id_mhs", "nama", "semester", "ipk"]].sort_values("nama").head(10))

df = df.drop(columns=["nama"])
print("Kolom nama dihapus. Jumlah ID unik:", df["id_mhs"].nunique())

## CELL 17 — III.3 Pembersihan: Duplikat, Tipe Data, & Validasi Rentang
*BAB III - PREPROCESS*

In [ ]:
cek_dup = df.drop(columns=["id_mhs", "timestamp"]).duplicated().sum()
print("Duplikat (isi jawaban identik semua kolom):", cek_dup)
if cek_dup:
    df = df.drop_duplicates(subset=df.columns.difference(["id_mhs", "timestamp"])).reset_index(drop=True)

kol_total = [f"total_{h}" for h in HARI]
kol_jam   = ["jam_kuliah", "jam_belajar", "jam_organisasi", "jam_kerja", "jam_lain"]
for c in ["semester", "ipk"] + kol_total + kol_jam:
    df[c] = pd.to_numeric(df[c], errors="coerce")

validasi = {
    "IPK di luar 0-4": (~df["ipk"].between(0, 4)).sum(),
    "Semester di luar 1-14": (~df["semester"].between(1, 14)).sum(),
    "Pengeluaran negatif": (df[kol_total] < 0).sum().sum(),
    "Jam negatif": (df[kol_jam] < 0).sum().sum(),
    "Total jam > 168/minggu": (df[kol_jam].sum(axis=1) > 168).sum(),
    "Missing value tersisa": df.drop(columns=["kampus"]).isna().sum().sum(),
}
display(pd.Series(validasi, name="jumlah_masalah").to_frame())

## CELL 18 — III.3b Deteksi & Penanganan Isian “Belum Hari” (Hari Belum Terjadi)
*BAB III - PREPROCESS*

In [ ]:
# Responden yang mengisi hari yang belum terjadi (mis. "blom hari rabu") dengan total Rp0
pola_belum = r"(?:belum|blm|blom|masih)\s+hari|kan\s+blom"
kol_rincian = [f"rincian_{h}" for h in HARI]

mask_belum = df[kol_rincian].apply(
    lambda s: s.astype(str).str.contains(pola_belum, case=False, regex=True)).any(axis=1)
print("Responden dengan isian 'belum hari':", int(mask_belum.sum()))
display(df.loc[mask_belum, ["id_mhs", "ipk"] + kol_total + kol_rincian])

df_dikeluarkan = df[mask_belum].copy()
df = df[~mask_belum].reset_index(drop=True)
print("Responden sebelum :", len(df) + len(df_dikeluarkan))
print("Responden sesudah :", len(df))

## CELL 19 — III.4 Parsing Rincian Pengeluaran (Teks Bebas -> Item, Nominal, Kategori)
*BAB III - PREPROCESS*

In [ ]:
KATEGORI_RULES = [
    ("bensin",          r"bensin|pertamax|pertalite|\boli\b|servis|tambal|\bban\b|\brem\b"),
    ("transportasi",    r"transport|tranport|trasnport|angkot|parkir|ojek|ongkos|tiket|grab|gojek"),
    ("pulsa_langganan", r"pulsa|kuota|paket data|internet|netflix|langganan|token|\bai\b|\bdata\b|spotify"),
    ("akademik",        r"fotokopi|print|cetak|alat tulis|pulpen|buku|materi|tugas|makalah|jilid|"
                        r"modul|laporan|maket|poster|rangkuman|kerkom|lomba"),
    ("makan",           r"makan|mkan|mkn|nasi|sarapan|ayam|warteg|geprek|soto|bakso|mie|bubur|"
                        r"padang|penyetan|pecel|lele|nasgor|telur|konsumsi"),
    ("minum_ngopi",     r"minum|minu\b|kopi|ngopi|\bes\b|teh|boba|air mineral|galon|cafe|cincau|jeruk|kelapa"),
    ("jajan",           r"jajan|camilan|cemilan|snack|nongkrong|martabak|roti|gorengan|donat|"
                        r"batagor|kerupuk|saset|rokok|nonton|bioskop|kentang|buah"),
]
KATEGORI = [k for k, _ in KATEGORI_RULES] + ["lainnya"]
NUM = re.compile(r"(\d{1,3}(?:\.\d{3})+|\d+)\s*(k|rb|ribu)?(?![a-z0-9])", re.I)

def tentukan_kategori(teks):
    t = teks.lower()
    for nama, pola in KATEGORI_RULES:
        if re.search(pola, t):
            return nama
    return "lainnya"

def parse_rincian(teks):
    hasil = []
    if not isinstance(teks, str):
        return hasil
    for seg in re.split(r"[\n;,]", teks):
        seg = seg.strip()
        if not seg:
            continue
        m = NUM.search(seg)
        if not m:
            continue
        angka = int(m.group(1).replace(".", ""))
        if m.group(2):
            angka *= 1000
        hasil.append((seg, angka, tentukan_kategori(seg)))
    return hasil

print(parse_rincian("100k ganti oli , 25k makan minum , 25k bensin"))
print(parse_rincian("Makanan: 40.000, Minuman: 30.000, Tranportasi. 30.000"))
print(parse_rincian("Tidak ada pengeluaran"))

## CELL 20 — III.4 Membangun Tabel Item Pengeluaran (Format Long)
*BAB III - PREPROCESS*

In [ ]:
baris = []
for _, r in df.iterrows():
    for h in HARI:
        for item, nominal, kat in parse_rincian(r[f"rincian_{h}"]):
            baris.append({"id_mhs": r["id_mhs"], "hari": h, "item": item,
                          "nominal": nominal, "kategori": kat})
df_item = pd.DataFrame(baris)
print("Jumlah item terparsing:", len(df_item))
display(df_item.head(10))

ringkas = (df_item.groupby("kategori")["nominal"]
           .agg(["count", "sum"]).reindex(KATEGORI).fillna(0).astype(int))
ringkas["persen_rupiah"] = (ringkas["sum"] / ringkas["sum"].sum() * 100).round(1)
display(ringkas)

## CELL 21 — III.5 Validasi Konsistensi Total Harian vs Jumlah Rincian
*BAB III - PREPROCESS*

In [ ]:
parsed = (df_item.groupby(["id_mhs", "hari"])["nominal"].sum()
          .unstack("hari").reindex(columns=HARI))
parsed = parsed.reindex(df["id_mhs"]).fillna(0)

total = df.set_index("id_mhs")[kol_total]
total.columns = HARI
selisih = total - parsed

cocok = (selisih.abs() <= np.maximum(1000, 0.05 * total)).values
print(f"Hari-responden konsisten (toleransi 5%/Rp1.000): {cocok.mean()*100:.1f}%")
print("Rata-rata selisih absolut (Rp):", int(selisih.abs().values.mean()))

tidak_cocok = (selisih.abs() > np.maximum(1000, 0.05 * total)).sum().sum()
print("Jumlah sel tidak konsisten:", tidak_cocok)
print("Keputusan: nilai TOTAL dipakai sebagai acuan, rincian dipakai untuk kategori.")

## CELL 22 — III.6 Feature Engineering: Agregat Pengeluaran Mingguan
*BAB III - PREPROCESS*

In [ ]:
df["pengeluaran_mingguan"] = df[kol_total].sum(axis=1)
df["rata_harian"]          = df[kol_total].mean(axis=1)
df["std_harian"]           = df[kol_total].std(axis=1)
df["hari_tanpa_pengeluaran"] = (df[kol_total] == 0).sum(axis=1)
df["pengeluaran_weekend"]  = df[["total_sabtu", "total_minggu"]].sum(axis=1)
df["jam_total_aktivitas"]  = df[kol_jam].sum(axis=1)
df["rasio_belajar_kuliah"] = df["jam_belajar"] / df["jam_kuliah"]

display(df[["id_mhs", "ipk", "pengeluaran_mingguan", "rata_harian",
            "hari_tanpa_pengeluaran", "jam_total_aktivitas"]].head())

## CELL 23 — III.6 Feature Engineering: Kategori Pengeluaran & Tipe Dominan
*BAB III - PREPROCESS*

In [ ]:
pivot = (df_item.pivot_table(index="id_mhs", columns="kategori", values="nominal",
                             aggfunc="sum", fill_value=0)
         .reindex(columns=KATEGORI, fill_value=0))
pivot = pivot.reindex(df["id_mhs"]).fillna(0)
pivot.columns = [f"rp_{c}" for c in pivot.columns]

prop = pivot.div(pivot.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
prop.columns = [c.replace("rp_", "prop_") for c in prop.columns]

df = df.merge(pivot.reset_index(), on="id_mhs").merge(prop.reset_index(), on="id_mhs")
df["kategori_dominan"] = (pivot.idxmax(axis=1).str.replace("rp_", "")
                          .where(pivot.sum(axis=1) > 0, "tidak_ada").values)

# kategori khas = kategori yang proporsinya paling melebihi rata-rata populasi
lift = prop - prop.mean()
df["kategori_khas"] = lift.idxmax(axis=1).str.replace("prop_", "").values

display(pd.concat([df["kategori_dominan"].value_counts().rename("dominan_rupiah"),
                   df["kategori_khas"].value_counts().rename("khas")], axis=1).fillna(0).astype(int))

## CELL 24 — III.7 Penanganan Outlier (IQR Winsorizing)
*BAB III - PREPROCESS*

In [ ]:
q1, q3 = df["pengeluaran_mingguan"].quantile([.25, .75])
iqr = q3 - q1
batas_bawah = max(0, q1 - 1.5 * iqr)
batas_atas  = q3 + 1.5 * iqr
print(f"Batas outlier pengeluaran mingguan: {batas_bawah:,.0f} - {batas_atas:,.0f}")

mask_out = (df["pengeluaran_mingguan"] > batas_atas) | (df["pengeluaran_mingguan"] < batas_bawah)
print("Jumlah outlier:", mask_out.sum())
display(df.loc[mask_out, ["id_mhs", "ipk", "pengeluaran_mingguan"]])

df["pengeluaran_w"] = df["pengeluaran_mingguan"].clip(batas_bawah, batas_atas)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
sns.boxplot(x=df["pengeluaran_mingguan"], ax=ax[0], color="#F4B183")
ax[0].set_title("Sebelum Winsorizing")
sns.boxplot(x=df["pengeluaran_w"], ax=ax[1], color="#A9D18E")
ax[1].set_title("Sesudah Winsorizing")
plt.tight_layout()
plt.show()

## CELL 25 — III.8 Eksplorasi Lanjutan: Korelasi Antar Variabel
*BAB III - PREPROCESS*

In [ ]:
kol_korelasi = ["ipk", "pengeluaran_w", "rata_harian", "hari_tanpa_pengeluaran"] + kol_jam
plt.figure(figsize=(8.5, 6.5))
sns.heatmap(df[kol_korelasi].corr(), annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1)
plt.title("Matriks Korelasi")
plt.tight_layout()
plt.show()

prop_cols = [c for c in df.columns if c.startswith("prop_")]
display(df.groupby("kategori_khas")[["ipk", "pengeluaran_w"]].agg(["mean", "count"]).round(2))

## CELL 26 — III.9 Seleksi Fitur untuk Clustering
*BAB III - PREPROCESS*

In [ ]:
FITUR = ["pengeluaran_w", "ipk"]          # model utama (sumbu X Rp, sumbu Y IPK)
FITUR_AKTIVITAS = kol_jam                  # dipakai untuk profiling cluster
FITUR_LENGKAP = FITUR + FITUR_AKTIVITAS    # model pembanding (7 fitur)

korel = df[FITUR_LENGKAP].corr().abs()
tinggi = [(a, b, round(korel.loc[a, b], 2))
          for i, a in enumerate(FITUR_LENGKAP) for b in FITUR_LENGKAP[i + 1:]
          if korel.loc[a, b] > 0.8]
print("Pasangan fitur berkorelasi > 0.8:", tinggi if tinggi else "tidak ada")
display(df[FITUR_LENGKAP].var().round(2).to_frame("variansi").T)

print("Fitur model utama     :", FITUR)
print("Variabel profiling    :", FITUR_AKTIVITAS)
print("Fitur model pembanding:", FITUR_LENGKAP)

## CELL 27 — III.10 Transformasi: Standardisasi (Z-Score)
*BAB III - PREPROCESS*

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(df[FITUR])

scaler_full = StandardScaler()
X_full = scaler_full.fit_transform(df[FITUR_LENGKAP])

print("Sebelum scaling:")
display(df[FITUR].describe().loc[["mean", "std"]].round(2))
print("Sesudah scaling:")
display(pd.DataFrame(X, columns=FITUR).describe().loc[["mean", "std"]].round(2))

## CELL 28 — IV.1 Penentuan Jumlah Cluster (Elbow & Silhouette, K = 2..6)
*BAB IV - MODEL*

In [ ]:
def evaluasi_k(Xm, ks=range(2, 7)):
    out = []
    for k in ks:
        km = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit(Xm)
        out.append({"k": k, "inertia": km.inertia_,
                    "silhouette": silhouette_score(Xm, km.labels_),
                    "davies_bouldin": davies_bouldin_score(Xm, km.labels_),
                    "calinski_harabasz": calinski_harabasz_score(Xm, km.labels_)})
    return pd.DataFrame(out)

eval_k = evaluasi_k(X)
eval_k_full = evaluasi_k(X_full)
print("Model utama (Pengeluaran x IPK):"); display(eval_k.round(3))
print("Model pembanding (7 fitur):");      display(eval_k_full.round(3))

fig, ax = plt.subplots(2, 2, figsize=(12, 7))
for r, (nama, e) in enumerate([("Model Utama", eval_k), ("Model 7 Fitur", eval_k_full)]):
    ax[r, 0].plot(e["k"], e["inertia"], "o-")
    ax[r, 0].set_title(f"{nama} - Elbow (Inertia/WCSS)")
    ax[r, 1].plot(e["k"], e["silhouette"], "o-", color="#ED7D31")
    ax[r, 1].set_title(f"{nama} - Silhouette Score")
    for a in ax[r]:
        a.set_xlabel("K")
plt.tight_layout()
plt.show()

## CELL 29 — IV.1 Menetapkan K Final
*BAB IV - MODEL*

In [ ]:
K_OVERRIDE = None     # isi angka (mis. 3) untuk menetapkan K secara manual

# aturan: K terkecil yang silhouette-nya berada dalam 0.01 dari nilai maksimum
sil_max = eval_k["silhouette"].max()
K_AUTO = int(eval_k.loc[eval_k["silhouette"] >= sil_max - 0.01, "k"].min())
K_FINAL = K_OVERRIDE or K_AUTO
print("Silhouette maksimum :", round(sil_max, 4))
print("K final             :", K_FINAL)

## CELL 30 — IV.2 Perhitungan Manual K-Means (Euclidean Distance, Tiap Iterasi)
*BAB IV - MODEL*

In [ ]:
def kmeans_manual(Xm, k, seed, max_iter=100, verbose=False):
    rng = np.random.default_rng(seed)
    cent = Xm[rng.choice(len(Xm), k, replace=False)].copy()
    cent0 = cent.copy()
    for it in range(1, max_iter + 1):
        jarak = np.sqrt(((Xm[:, None, :] - cent[None, :, :]) ** 2).sum(axis=2))
        lab = jarak.argmin(axis=1)
        baru = np.array([Xm[lab == j].mean(axis=0) if (lab == j).any() else cent[j]
                         for j in range(k)])
        wcss = sum(((Xm[lab == j] - baru[j]) ** 2).sum() for j in range(k))
        if verbose:
            print(f"Iterasi {it:02d} | WCSS = {wcss:.4f}")
        if np.allclose(baru, cent):
            break
        cent = baru
    return lab, cent, wcss, it, cent0

# rincian iterasi untuk satu inisialisasi
_, _, _, _, cent_awal = kmeans_manual(X, K_FINAL, SEED, verbose=True)
print("\nCentroid awal (z-score):\n", cent_awal.round(4))

# jarak Euclidean 5 mahasiswa pertama ke centroid awal
jarak_awal = np.sqrt(((X[:5, None, :] - cent_awal[None, :, :]) ** 2).sum(axis=2))
display(pd.DataFrame(jarak_awal.round(4), index=df["id_mhs"][:5],
                     columns=[f"jarak_ke_C{j}" for j in range(K_FINAL)])
        .assign(cluster_awal=jarak_awal.argmin(axis=1)))

# hasil terbaik dari 30 inisialisasi acak
hasil = [kmeans_manual(X, K_FINAL, s) for s in range(30)]
lab_man, cent_man, wcss_man, it_man, _ = min(hasil, key=lambda h: h[2])
print(f"\nManual terbaik : WCSS={wcss_man:.4f}, iterasi={it_man}")

## CELL 31 — IV.3 Model Final K-Means (scikit-learn) & Perbandingan dengan Manual
*BAB IV - MODEL*

In [ ]:
kmeans = KMeans(n_clusters=K_FINAL, n_init=50, random_state=SEED).fit(X)
df["cluster"] = kmeans.labels_

print(f"sklearn : WCSS={kmeans.inertia_:.4f}, iterasi={kmeans.n_iter_}")
print(f"Kesesuaian manual vs sklearn (ARI): {adjusted_rand_score(lab_man, kmeans.labels_):.4f}")

pusat = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=FITUR).round(2)
pusat.index.name = "cluster"
print("Centroid (skala asli):")
display(pusat)
display(df["cluster"].value_counts().sort_index().to_frame("jumlah_mahasiswa"))

## CELL 32 — IV.4 Profil & Penamaan Cluster
*BAB IV - MODEL*

In [ ]:
prop_cols = [c for c in df.columns if c.startswith("prop_")]
fitur_profil = FITUR + FITUR_AKTIVITAS + ["semester", "hari_tanpa_pengeluaran"] + prop_cols
profil = df.groupby("cluster")[fitur_profil].mean().round(3)
profil.insert(0, "jumlah", df["cluster"].value_counts().sort_index())
profil["kategori_khas_terbanyak"] = df.groupby("cluster")["kategori_khas"].agg(
    lambda s: s.value_counts().index[0])
display(profil.T)

z = pd.DataFrame(kmeans.cluster_centers_, columns=FITUR)
z_akt = (profil[FITUR_AKTIVITAS] - df[FITUR_AKTIVITAS].mean()) / df[FITUR_AKTIVITAS].std()
label_akt = {"jam_belajar": "rajin belajar", "jam_organisasi": "aktif organisasi",
             "jam_kerja": "banyak bekerja"}

def beri_nama(c):
    r = z.loc[c]
    if r["ipk"] >= 0 and r["pengeluaran_w"] < 0:  dasar = "Hemat & Berprestasi"
    elif r["ipk"] >= 0:                           dasar = "Pengeluaran Tinggi & Berprestasi"
    elif r["pengeluaran_w"] >= 0:                 dasar = "Pengeluaran Tinggi, Prestasi Menurun"
    else:                                         dasar = "Hemat, Perlu Pendampingan"
    kuat = [label_akt[a] for a in label_akt if z_akt.loc[c, a] > 0.5]
    return dasar + (f" ({', '.join(kuat)})" if kuat else "")

nama_cluster = {c: beri_nama(c) for c in range(K_FINAL)}
for n in set(nama_cluster.values()):
    dup = [c for c, v in nama_cluster.items() if v == n]
    if len(dup) > 1:
        for c in dup:
            nama_cluster[c] = f"{n} - C{c}"
df["nama_cluster"] = df["cluster"].map(nama_cluster)
for c, n in nama_cluster.items():
    print(f"Cluster {c}: {n}")

## CELL 33 — IV.5 Visualisasi Cluster Sesuai Sketsa (X = Pengeluaran, Y = IPK)
*BAB IV - MODEL*

In [ ]:
from scipy.spatial import ConvexHull
from matplotlib.patches import Polygon

MARKER = {"bensin": "X", "transportasi": "P", "pulsa_langganan": "D", "akademik": "s",
          "makan": "o", "minum_ngopi": "^", "jajan": "v", "lainnya": "*"}
warna = sns.color_palette("Set1", K_FINAL)

fig, ax = plt.subplots(figsize=(10, 6.5))
for c in range(K_FINAL):
    pts = df.loc[df["cluster"] == c, ["pengeluaran_w", "ipk"]].values
    if len(pts) >= 3:
        h = ConvexHull(pts)
        ax.add_patch(Polygon(pts[h.vertices], closed=True, facecolor=warna[c],
                             alpha=.12, edgecolor=warna[c], lw=1.8))
for kat, mk in MARKER.items():
    for c in range(K_FINAL):
        s = df[(df["kategori_khas"] == kat) & (df["cluster"] == c)]
        ax.scatter(s["pengeluaran_w"], s["ipk"], marker=mk, color=warna[c], s=65,
                   edgecolor="k", linewidth=.4, alpha=.9,
                   label=kat if c == 0 else None)
ctr = scaler.inverse_transform(kmeans.cluster_centers_)
ax.scatter(ctr[:, 0], ctr[:, 1], marker="*", s=450, color="gold", edgecolor="k",
           label="centroid", zorder=5)
for c in range(K_FINAL):
    ax.annotate(f"C{c}", (ctr[c, 0], ctr[c, 1]), xytext=(8, 8),
                textcoords="offset points", fontweight="bold")
ax.set_xlabel("Pengeluaran per minggu (Rp)  [sumbu X]")
ax.set_ylabel("IPK  [sumbu Y]")
ax.set_title("Hasil K-Means: warna = cluster, bentuk = kategori pengeluaran khas")
ax.legend(title="Kategori khas", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/scatter_cluster.png", bbox_inches="tight")
plt.show()

## CELL 34 — IV.6 Profil Aktivitas Harian per Cluster
*BAB IV - MODEL*

In [ ]:
akt = df.groupby("nama_cluster")[FITUR_AKTIVITAS].mean()
akt.columns = ["Kuliah", "Belajar mandiri", "Organisasi", "Bekerja", "Kegiatan lain"]
akt.plot(kind="bar", figsize=(10, 5), colormap="Set2")
plt.ylabel("Rata-rata jam per minggu")
plt.title("Pola Aktivitas Mingguan per Cluster")
plt.xticks(rotation=15, ha="right")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()
display(akt.round(2))

## CELL 35 — IV.7 Model Pembanding: K-Means 7 Fitur (Pengeluaran + IPK + Aktivitas)
*BAB IV - MODEL*

In [ ]:
kmeans_full = KMeans(n_clusters=K_FINAL, n_init=50, random_state=SEED).fit(X_full)
df["cluster_full"] = kmeans_full.labels_

pusat_full = pd.DataFrame(scaler_full.inverse_transform(kmeans_full.cluster_centers_),
                          columns=FITUR_LENGKAP).round(2)
display(pusat_full)
display(df["cluster_full"].value_counts().sort_index().to_frame("jumlah"))
print("Kesesuaian cluster utama vs pembanding (ARI):",
      round(adjusted_rand_score(df["cluster"], df["cluster_full"]), 4))

## CELL 36 — V.1 Metrik Evaluasi Internal (Silhouette, Davies-Bouldin, Calinski-Harabasz)
*BAB V - EVALUASI*

In [ ]:
def ringkas_metrik(Xm, lab, inertia):
    return {"jumlah_cluster": len(set(lab)),
            "inertia_WCSS": round(inertia, 3),
            "silhouette": round(silhouette_score(Xm, lab), 4),
            "davies_bouldin": round(davies_bouldin_score(Xm, lab), 4),
            "calinski_harabasz": round(calinski_harabasz_score(Xm, lab), 2)}

metrik = pd.DataFrame({
    "Model Utama (Rp x IPK)": ringkas_metrik(X, kmeans.labels_, kmeans.inertia_),
    "Pembanding (7 fitur)":   ringkas_metrik(X_full, kmeans_full.labels_, kmeans_full.inertia_),
}).T
display(metrik)
print("Panduan: silhouette lebih tinggi lebih baik (>0.5 kuat, 0.25-0.5 sedang, <0.25 lemah);")
print("         Davies-Bouldin lebih rendah lebih baik; Calinski-Harabasz lebih tinggi lebih baik.")

## CELL 37 — V.2 Silhouette Plot per Cluster
*BAB V - EVALUASI*

In [ ]:
sil = silhouette_samples(X, kmeans.labels_)
fig, ax = plt.subplots(figsize=(8, 5))
y0 = 10
for c in range(K_FINAL):
    s = np.sort(sil[kmeans.labels_ == c])
    ax.fill_betweenx(np.arange(y0, y0 + len(s)), 0, s, alpha=.8, label=nama_cluster[c])
    y0 += len(s) + 10
ax.axvline(sil.mean(), color="red", ls="--", label=f"rata-rata={sil.mean():.3f}")
ax.set_xlabel("Nilai silhouette"); ax.set_yticks([])
ax.set_title("Silhouette Plot per Cluster")
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

df["silhouette_i"] = sil
display(df.groupby("nama_cluster")["silhouette_i"].agg(["mean", "min", "count"]).round(3))
print("Mahasiswa dengan silhouette negatif (berpotensi salah cluster):", int((sil < 0).sum()))

## CELL 38 — V.3 Uji Stabilitas Cluster (Seed Berbeda & Bootstrap, ARI)
*BAB V - EVALUASI*

In [ ]:
ari_seed = []
for s in range(100, 130):
    lab = KMeans(n_clusters=K_FINAL, n_init=1, random_state=s).fit_predict(X)
    ari_seed.append(adjusted_rand_score(kmeans.labels_, lab))

rng = np.random.default_rng(SEED)
ari_boot = []
for _ in range(100):
    idx = rng.choice(len(X), len(X), replace=True)
    km_b = KMeans(n_clusters=K_FINAL, n_init=10, random_state=SEED).fit(X[idx])
    ari_boot.append(adjusted_rand_score(kmeans.labels_, km_b.predict(X)))

print(f"ARI antar seed (30x) : mean={np.mean(ari_seed):.3f}, min={np.min(ari_seed):.3f}")
print(f"ARI bootstrap (100x) : mean={np.mean(ari_boot):.3f}, min={np.min(ari_boot):.3f}")

plt.figure(figsize=(7, 3.5))
sns.histplot(ari_boot, bins=15, color="#5B9BD5")
plt.xlabel("ARI (bootstrap vs model final)")
plt.title("Distribusi Stabilitas Cluster")
plt.show()

## CELL 39 — V.4 Perbandingan dengan Algoritma Lain (Hierarchical Ward)
*BAB V - EVALUASI*

In [ ]:
agg = AgglomerativeClustering(n_clusters=K_FINAL, linkage="ward").fit(X)
banding = pd.DataFrame({
    "K-Means": [silhouette_score(X, kmeans.labels_),
                davies_bouldin_score(X, kmeans.labels_),
                calinski_harabasz_score(X, kmeans.labels_)],
    "Agglomerative (Ward)": [silhouette_score(X, agg.labels_),
                             davies_bouldin_score(X, agg.labels_),
                             calinski_harabasz_score(X, agg.labels_)],
}, index=["silhouette", "davies_bouldin", "calinski_harabasz"]).round(4)
display(banding)
print("Kesesuaian K-Means vs Ward (ARI):", round(adjusted_rand_score(kmeans.labels_, agg.labels_), 4))

## CELL 40 — V.5 Uji Signifikansi Perbedaan Antar Cluster (Kruskal-Wallis)
*BAB V - EVALUASI*

In [ ]:
uji = []
for f in FITUR + FITUR_AKTIVITAS + ["semester", "hari_tanpa_pengeluaran"]:
    grup = [df.loc[df["cluster"] == c, f] for c in range(K_FINAL)]
    h, p = stats.kruskal(*grup)
    uji.append({"variabel": f, "H": round(h, 2), "p_value": round(p, 4),
                "signifikan_0.05": p < 0.05})
display(pd.DataFrame(uji))

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
sns.boxplot(x="nama_cluster", y="ipk", data=df, palette="Set2", ax=ax[0])
ax[0].set_title("Sebaran IPK per Cluster")
sns.boxplot(x="nama_cluster", y="pengeluaran_mingguan", data=df, palette="Set3", ax=ax[1])
ax[1].set_title("Sebaran Pengeluaran Mingguan per Cluster")
for a in ax:
    a.tick_params(axis="x", rotation=20)
    plt.setp(a.get_xticklabels(), ha="right", fontsize=7)
plt.tight_layout()
plt.show()

## CELL 41 — V.6 Komposisi Kategori Pengeluaran per Cluster
*BAB V - EVALUASI*

In [ ]:
komposisi = df.groupby("nama_cluster")[prop_cols].mean()
komposisi.columns = [c.replace("prop_", "") for c in komposisi.columns]
komposisi.plot(kind="barh", stacked=True, figsize=(10, 4 + 0.4 * K_FINAL), colormap="tab20")
plt.xlabel("Proporsi rupiah")
plt.title("Komposisi Kategori Pengeluaran per Cluster")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## CELL 42 — V.7 Ringkasan Hasil Evaluasi
*BAB V - EVALUASI*

In [ ]:
ringkasan_eval = {
    "K_final": K_FINAL,
    "silhouette": round(silhouette_score(X, kmeans.labels_), 4),
    "davies_bouldin": round(davies_bouldin_score(X, kmeans.labels_), 4),
    "calinski_harabasz": round(calinski_harabasz_score(X, kmeans.labels_), 2),
    "ARI_stabilitas_seed_mean": round(float(np.mean(ari_seed)), 4),
    "ARI_stabilitas_bootstrap_mean": round(float(np.mean(ari_boot)), 4),
    "variabel_beda_signifikan": [u["variabel"] for u in uji if u["signifikan_0.05"]],
}
for k, v in ringkasan_eval.items():
    print(f"{k:32s}: {v}")

## CELL 43 — VI.1 Menyimpan Model & Artefak Analisis
*BAB VI - PENERAPAN*

In [ ]:
def buat_saran(nama):
    if nama.startswith("Hemat & Berprestasi"):
        return "Pola keuangan dan akademik sudah baik. Pertahankan, dan dapat dilibatkan sebagai mentor sebaya."
    if nama.startswith("Pengeluaran Tinggi & Berprestasi"):
        return "Prestasi baik namun pengeluaran tinggi. Disarankan menyusun anggaran mingguan."
    if nama.startswith("Pengeluaran Tinggi, Prestasi Menurun"):
        return ("Pengeluaran tinggi dengan IPK di bawah rata-rata. "
                "Prioritas pembinaan manajemen keuangan dan waktu belajar.")
    return ("Pengeluaran rendah namun IPK di bawah rata-rata. "
            "Perlu pendampingan akademik dan evaluasi beban aktivitas.")

saran = {int(c): buat_saran(n) for c, n in nama_cluster.items()}

artefak = {
    "scaler": scaler, "kmeans": kmeans, "fitur": FITUR,
    "nama_cluster": {int(k): v for k, v in nama_cluster.items()},
    "saran": saran,
    "cap_pengeluaran": float(batas_atas),
    "profil": profil,
    "profil_aktivitas": df.groupby("cluster")[FITUR_AKTIVITAS].mean().round(2),
}
joblib.dump(artefak, f"{OUT_DIR}/artefak_model.joblib")
df[["id_mhs", "ipk", "pengeluaran_mingguan", "pengeluaran_w", "cluster", "nama_cluster"]] \
    .to_csv(f"{OUT_DIR}/data_cluster.csv", index=False)
print("Tersimpan:", os.listdir(OUT_DIR))

## CELL 44 — VI.2 Simulasi Prediksi Cluster Mahasiswa Baru
*BAB VI - PENERAPAN*

In [ ]:
def prediksi_cluster(ipk, pengeluaran_harian):
    art = joblib.load(f"{OUT_DIR}/artefak_model.joblib")
    total = min(sum(pengeluaran_harian), art["cap_pengeluaran"])
    x = pd.DataFrame([[total, ipk]], columns=art["fitur"])
    c = int(art["kmeans"].predict(art["scaler"].transform(x))[0])
    return c, art["nama_cluster"][c], art["saran"][c]

contoh = prediksi_cluster(3.4, [30000, 40000, 35000, 30000, 45000, 20000, 25000])
print("Cluster :", contoh[0], "-", contoh[1])
print("Saran   :", contoh[2])

## CELL 45 — VI.3 Kesimpulan Otomatis Hasil Clustering
*BAB VI - PENERAPAN*

In [ ]:
print(f"Jumlah responden dianalisis : {len(df)}")
print(f"Jumlah cluster (K-Means)    : {K_FINAL}")
print(f"Silhouette Score            : {ringkasan_eval['silhouette']}")
print()
for c in range(K_FINAL):
    s = df[df["cluster"] == c]
    print(f"Cluster {c} - {nama_cluster[c]}")
    print(f"  jumlah         : {len(s)} ({len(s)/len(df)*100:.1f}%)")
    print(f"  rata-rata IPK  : {s['ipk'].mean():.2f}")
    print(f"  pengeluaran/mg : Rp{s['pengeluaran_mingguan'].mean():,.0f}")
    print(f"  jam belajar    : {s['jam_belajar'].mean():.1f} | organisasi: {s['jam_organisasi'].mean():.1f}"
          f" | kerja: {s['jam_kerja'].mean():.1f}")
    print(f"  kategori khas  : {s['kategori_khas'].value_counts().index[0]}")
    print(f"  saran          : {saran[c]}")
    print()

## CELL 46 — Lampiran: Cuplikan Data Mentah & Ekspor Seluruh Hasil
*LAMPIRAN*

In [ ]:
cuplikan = df_raw.drop(columns=["Nama(bisa samaran)", "Kolom 20"]).head(10)
display(cuplikan)

with pd.ExcelWriter(f"{OUT_DIR}/hasil_clustering.xlsx") as w:
    cuplikan.to_excel(w, sheet_name="cuplikan_data_mentah", index=False)
    df.drop(columns=[c for c in df.columns if c.startswith("rincian_")]) \
      .to_excel(w, sheet_name="data_bersih_cluster", index=False)
    df_item.to_excel(w, sheet_name="item_pengeluaran", index=False)
    df_dikeluarkan.to_excel(w, sheet_name="data_dikeluarkan", index=False)
    profil.T.to_excel(w, sheet_name="profil_cluster")
    metrik.to_excel(w, sheet_name="metrik_evaluasi")
print("File hasil:", os.listdir(OUT_DIR))

## CELL 47 — Lampiran: Unduh Semua Hasil (ZIP)
*LAMPIRAN*

In [ ]:
import shutil
import os

ZIP_PATH = "/content/hasil_project.zip"

# Buat ulang ZIP dari seluruh isi folder hasil, termasuk semua PNG.
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

shutil.make_archive("/content/hasil_project", "zip", OUT_DIR)

print("Jumlah file PNG:", sum(
    1 for nama in os.listdir(PNG_DIR) if nama.lower().endswith(".png")
))
print("File ZIP berhasil dibuat:", ZIP_PATH)

try:
    from google.colab import files
    files.download(ZIP_PATH)
except ImportError:
    print("Unduh manual dari:", ZIP_PATH)
